# 客户端概况
了解如何使用 FastMCP 客户端与 MCP 服务器交互。

它`fastmcp.Client`提供了一个高级异步接口，用于与任何模型上下文协议 (MCP) 服务器交互，无论该服务器是基于 FastMCP 构建的还是其他实现。它通过处理协议细节和连接管理来简化通信。


## FastMCP 客户端
FastMCP 客户端架构将协议逻辑`（Client）`与连接机制`（Transport）`分离。

- `Client`：处理发送 MCP 请求（如`tools/call`、`resources/read`）、接收响应和管理回调。
- `Transport`：负责建立和维护与服务器的连接（例如，通过 `WebSockets`、`SSE`、`Stdio` 或`内存`）。

## transport

客户端必须使用`transport`进行初始化。您可以提供一个已经实例化的传输对象，或者提供一个传输源，让 FastMCP 尝试推断要使用的正确传输方式。

以下推理规则用于`ClientTransport`根据输入类型确定适当的：

-  `ClientTransport`实例：如果您提供已经实例化的传输对象，则直接使用它。
-  `FastMCP实例`：创建一个FastMCPTransport高效的内存通信（适合测试）。
- Path或str指向现有文件：
    - 如果它以以下内容结尾.py：创建一个PythonStdioTransport以使用运行脚本python。
    - 如果它以以下内容结尾.js：创建一个NodeStdioTransport以使用运行脚本node。
- `AnyUrl`或str指向以http://或https://开头的 URL ：
- 创建一个`StreamableHttpTransport`
- 其他：如果无法推断类型，则引发`ValueError`。

In [ ]:
import asyncio
from fastmcp import Client, FastMCP

# Example transports (more details in Transports page)
server_instance = FastMCP(name="TestServer") # In-memory server
http_url = "https://example.com/mcp"        # HTTP server URL
ws_url = "ws://localhost:9000"             # WebSocket server URL
server_script = "my_mcp_server.py"         # Path to a Python server file

# Client automatically infers the transport type
client_in_memory = Client(server_instance)
client_http = Client(http_url)
client_ws = Client(ws_url)
client_stdio = Client(server_script)

print(client_in_memory.transport)
print(client_http.transport)
print(client_ws.transport)
print(client_stdio.transport)

# Expected Output (types may vary slightly based on environment):
# <FastMCP(server='TestServer')>
# <StreamableHttp(url='https://example.com/mcp')>
# <WebSocket(url='ws://localhost:9000')>
# <PythonStdioTransport(command='python', args=['/path/to/your/my_mcp_server.py'])>

>为了更好地控制连接细节（例如 SSE 的标头、Stdio 的环境变量），您可以ClientTransport自行实例化特定类并将其传递给Client。详情请参阅传输页面。

## 客户端使用情况
​
### 连接生命周期
客户端异步操作，必须在块内使用`async with`。上下文管理器负责建立连接、初始化 MCP 会话以及退出时清理资源。

In [ ]:
import asyncio
from fastmcp import Client

client = Client("my_mcp_server.py") # Assumes my_mcp_server.py exists

async def main():
    # Connection is established here
    async with client:
        print(f"Client connected: {client.is_connected()}")

        # Make MCP calls within the context
        tools = await client.list_tools()
        print(f"Available tools: {tools}")

        if any(tool.name == "greet" for tool in tools):
            result = await client.call_tool("greet", {"name": "World"})
            print(f"Greet result: {result}")

    # Connection is closed automatically here
    print(f"Client connected: {client.is_connected()}")

if __name__ == "__main__":
    asyncio.run(main())

### 客户端方法
提供Client与标准 MCP 请求对应的方法：

> 标准客户端方法返回用户友好的表示形式，这些表示形式可能会随着协议的发展而变化。为了能够一致地访问完整的数据结构，请使用`*_mcp`稍后介绍的方法。


### 工具操作
`list_tools()`：检索服务器上可用工具的列表。

In [ ]:
tools = await client.list_tools()
# tools -> list[mcp.types.Tool]

`call_tool(name: str, arguments: dict[str, Any] | None = None, timeout: float | None = None)`：在服务器上执行工具。

In [ ]:
result = await client.call_tool("add", {"a": 5, "b": 3})
# result -> list[mcp.types.TextContent | mcp.types.ImageContent | ...]
print(result[0].text) # Assuming TextContent, e.g., '8'

# With timeout (aborts if execution takes longer than 2 seconds)
result = await client.call_tool("long_running_task", {"param": "value"}, timeout=2.0)

- 参数以字典形式传递。如有需要，FastMCP 服务器会自动处理复杂类型的 JSON 字符串解析。
- 返回内容对象列表（通常`TextContent`为 或`ImageContent`）。
- 可选`timeout`参数限制此特定调用的最大执行时间（以秒为单位），覆盖任何客户端级超时。

### 资源操作
`list_resources()`：检索静态资源列表。

In [ ]:
templates = await client.list_resource_templates()
# templates -> list[mcp.types.ResourceTemplate]

`read_resource(uri: str | AnyUrl)`：读取资源或已解析模板的内容。

In [ ]:
# Read a static resource
readme_content = await client.read_resource("file:///path/to/README.md")
# readme_content -> list[mcp.types.TextResourceContents | mcp.types.BlobResourceContents]
print(readme_content[0].text) # Assuming text

# Read a resource generated from a template
weather_content = await client.read_resource("data://weather/london")
print(weather_content[0].text) # Assuming text JSON

### 快捷操作
- `list_prompts()`：检索可用的提示模板。
- `get_prompt(name: str, arguments: dict[str, Any] | None = None)`：检索呈现的提示消息列表。

## 原始 MCP 协议对象

FastMCP 客户端试图提供一个“友好”的 MCP 协议接口，但有时你可能需要访问原始的 MCP 协议对象。每个返回数据的主要客户端方法都有一个对应的*_mcp方法，可以直接返回原始的 MCP 协议对象。

In [ ]:
# Standard method - returns just the list of tools
tools = await client.list_tools()
# tools -> list[mcp.types.Tool]

# Raw MCP method - returns the full protocol object
result = await client.list_tools_mcp()
# result -> mcp.types.ListToolsResult
tools = result.tools

可用的原始 MCP 方法：

- list_tools_mcp()：返回 mcp.types.ListToolsResult
- call_tool_mcp(name, arguments)：返回 mcp.types.CallToolResult
- list_resources_mcp()：返回 mcp.types.ListResourcesResult
- list_resource_templates_mcp()：返回 mcp.types.ListResourceTemplatesResult
- read_resource_mcp(uri)：返回 mcp.types.ReadResourceResult
- list_prompts_mcp()：返回 mcp.types.ListPromptsResult
- get_prompt_mcp(name, arguments)：返回 mcp.types.GetPromptResult
- complete_mcp(ref, argument)：返回 mcp.types.CompleteResult

这些方法对于调试或需要访问简化方法未公开的元数据或字段时特别有用。

## 高级功能
MCP 允许服务器与客户端交互，以提供额外的功能。Client构造函数接受额外的配置来处理这些服务器请求。

### 超时控制

您可以在客户端级别和单个请求级别控制请求超时：

In [ ]:
from fastmcp import Client
from fastmcp.exceptions import McpError

# Client with a global 5-second timeout for all requests
client = Client(
    my_mcp_server,
    timeout=5.0  # Default timeout in seconds
)

async with client:
    # This uses the global 5-second timeout
    result1 = await client.call_tool("quick_task", {"param": "value"})
    
    # This specifies a 10-second timeout for this specific call
    result2 = await client.call_tool("slow_task", {"param": "value"}, timeout=10.0)
    
    try:
        # This will likely timeout
        result3 = await client.call_tool("medium_task", {"param": "value"}, timeout=0.01)
    except McpError as e:
        # Handle timeout error
        print(f"The task timed out: {e}")

## LLM抽样
MCP 服务器可以向客户端请求 LLM 补全。客户端可以提供一个`sampling_handler`来处理这些请求。采样处理程序从服务器接收消息列表和其他参数，并返回一个字符串补全。

以下示例使用marvin该库来生成完成：

In [ ]:
import marvin
from fastmcp import Client
from fastmcp.client.sampling import (
                    SamplingMessage,
                    SamplingParams,
                    RequestContext,
                )

async def sampling_handler(
    messages: list[SamplingMessage],
    params: SamplingParams,
    context: RequestContext
) -> str:
    return await marvin.say_async(
        message=[m.content.text for m in messages],
        instructions=params.systemPrompt,
    )

client = Client(
    ...,
    sampling_handler=sampling_handler,
)

### 日志记录
MCP 服务器可以向客户端发送日志。客户端可以设置日志回调来接收这些日志。


In [ ]:
from fastmcp import Client
from fastmcp.client.logging import LogHandler, LogMessage

async def my_log_handler(params: LogMessage):
    print(f"[Server Log - {params.level.upper()}] {params.logger or 'default'}: {params.data}")

client_with_logging = Client(
    ...,
    log_handler=my_log_handler,
)

### Roots

根是客户端用来告知服务器其可访问的资源或访问权限限制的一种方式。服务器可以使用这些信息来调整行为或提供更准确的响应。

服务器可以向客户端请求根，客户端可以在其根发生改变时通知服务器。

要在创建客户端时设置根，用户可以提供根列表（可以是字符串列表）或返回根列表的异步函数。

In [ ]:
### 静态

from fastmcp import Client

client = Client(
    ..., 
    roots=["/path/to/root1", "/path/to/root2"],
)

In [ ]:
### 动态

from fastmcp import Client
from fastmcp.client.roots import RequestContext

async def roots_callback(context: RequestContext) -> list[str]:
    print(f"Server requested roots (Request ID: {context.request_id})")
    return ["/path/to/root1", "/path/to/root2"]

client = Client(
    ..., 
    roots=roots_callback,
)



## 实用方法

 - ping()：向服务器发送 ping 请求以验证连接。

In [ ]:
async def check_connection():
    async with client:
        await client.ping()
        print("Server is reachable")

## 错误处理

当call_tool请求导致服务器出现错误（例如，工具函数引发异常）时，该client.call_tool()方法将引发fastmcp.client.ClientError。

In [ ]:
async def safe_call_tool():
    async with client:
        try:
            # Assume 'divide' tool exists and might raise ZeroDivisionError
            result = await client.call_tool("divide", {"a": 10, "b": 0})
            print(f"Result: {result}")
        except ClientError as e:
            print(f"Tool call failed: {e}")
        except ConnectionError as e:
            print(f"Connection failed: {e}")
        except Exception as e:
            print(f"An unexpected error occurred: {e}")

# Example Output if division by zero occurs:
# Tool call failed: Division by zero is not allowed.

>  其他错误（例如连接失败）将引发标准 Python 异常（例如ConnectionError，TimeoutError）。